# 00g — Assemble sensor-derived candidates

Return to the graph environment. Run after 00e and 00f have finished the full cohort. Requires `candidate_preparation.py` beside the notebook.

The parser extracts a small, prespecified clinical vocabulary with polarity and original quoted text. It is a pilot rule parser, not a validated clinical entity extractor. Review its concepts/negation on training/validation data; freeze any changes before test use. Reports with no recognized concepts yield an explicit empty candidate list, counted in the audit. Failed or missing generation prevents a new candidates.json from being written. Clinical facts are never inferred from admission pairing.

This preserves the provenance of candidates and keeps independent annotations separate.

In [ ]:
from pathlib import Path
import json, sys
NOTEBOOK_DIR=Path.cwd().resolve()
PROJECT_ROOT=next((p for p in (NOTEBOOK_DIR,*NOTEBOOK_DIR.parents) if (p/'config.json').is_file()),None)
if PROJECT_ROOT is None:raise FileNotFoundError('Start in graph_reasoning or its extraction subfolder; project config.json is required')
HELPER_ROOT=next((p for p in (NOTEBOOK_DIR,PROJECT_ROOT,PROJECT_ROOT/'symile_extraction_update') if (p/'candidate_preparation.py').is_file() and (p/'sensor_candidate_generation.py').is_file()),None)
if HELPER_ROOT is None:raise FileNotFoundError('Copy both preparation helpers into the project or extraction subfolder')
sys.path.insert(0,str(HELPER_ROOT))
from candidate_preparation import *
CONFIG=read(PROJECT_ROOT/'config.json')
def resolve(value):
    p=Path(value).expanduser()
    return (p if p.is_absolute() else PROJECT_ROOT/p).resolve()
SOURCES=CONFIG.get('export_sources',{})
MANIFEST=resolve(SOURCES.get('manifest','data/prepared/manifest.json'))
SENSORS=MANIFEST.parent/'sensor_sources.json'
ARTIFACT_ROOT=resolve(CONFIG.get('embedding_artifact_root','/path/to/data/symile/ecg_cxr_bridge_artifacts'))
GENERATION_DIR=resolve(CONFIG.get('candidate_generation_dir','outputs/symile_sensor_only_v1'))
assert MANIFEST.is_file() and SENSORS.is_file(), 'Run notebook 00b first'


In [ ]:
CANDIDATES=resolve(SOURCES.get('candidates','data/prepared/candidates.json'))
summary=assemble_candidates(MANIFEST,SENSORS,GENERATION_DIR,CANDIDATES)
print(summary)
print('Saved:',CANDIDATES)
for name in ('manifest','embeddings','candidates','annotations'):
    path=resolve(SOURCES.get(name,f'data/prepared/{name}.json'))
    print(name,'exists:',path.is_file())
print('When all four files exist, run notebook 01 and wait for READY, then run notebook 02.')
